In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as sp_stats
from scipy.stats import binomtest, mannwhitneyu, fisher_exact, kruskal
from IPython.display import display, HTML, Markdown

# ── Database connection ──
DB_PATH = "C:/Users/scgee/OneDrive/Documents/Projects/PatientPunk/data/abortion_1month.db"
conn = sqlite3.connect(DB_PATH)

# ── Sentiment mapping ──
SENTIMENT_SCORE = {"positive": 1.0, "mixed": 0.5, "neutral": 0.0, "negative": -1.0}

def to_numeric(s):
    """Convert sentiment string to numeric score."""
    return SENTIMENT_SCORE.get(s, 0.0)

def classify_outcome(avg_score):
    """Classify user-level average into outcome category."""
    if avg_score > 0.7:
        return "positive"
    elif avg_score < -0.3:
        return "negative"
    return "mixed/neutral"

def wilson_ci(k, n, z=1.96):
    """Wilson score confidence interval for a proportion."""
    if n == 0:
        return 0.0, 0.0
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    margin = z * np.sqrt((p * (1 - p) + z**2 / (4 * n)) / n) / denom
    return max(0, center - margin), min(1, center + margin)

def nnt(treatment_rate, baseline_rate):
    """Number needed to treat. Returns None if rates are equal or inverted."""
    diff = treatment_rate - baseline_rate
    if diff <= 0:
        return None
    return round(1 / diff, 1)

# ── Chart defaults ──
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

# ── Filtering sets ──
GENERIC_TERMS = {
    "supplements", "medication", "treatment", "therapy", "drug", "drugs",
    "vitamin", "prescription", "pill", "pills", "dosage", "dose",
}

# Colors
COLORS = {"positive": "#2ecc71", "mixed/neutral": "#95a5a6", "negative": "#e74c3c"}


**Research Question:** "What predicts a positive vs negative abortion experience? Is it the method, the support system, or the clinical environment?"

## Abstract

In a 1-month sample of r/abortion (2,052 users, 9,885 posts), we use text-based theme extraction to classify abortion experiences as positive or negative and identify predictors via logistic regression. Three domains are analyzed: **method** (medical vs surgical), **support system** (partner, family, friend, alone), and **clinical environment** (staff quality, setting, sedation). The key finding is counterintuitive: partner involvement correlates with *worse* reported experiences (OR=0.46, p=0.0002), while friend support correlates with better outcomes. Clinical environment variables -- particularly positive staff interactions and sedation -- are the strongest positive predictors. Method (medical vs surgical) shows no significant independent effect after controlling for support and clinical factors. **Gestational age is not controlled for in this analysis**, which is a critical limitation. These findings suggest that the *social and clinical context* of an abortion matters more than the procedure type for predicting patient-reported experience quality.

## Data Landscape

This is a community experience analysis. The richest signal in r/abortion is in post text, not treatment reports. Users share detailed narratives about their experiences -- who was with them, how staff treated them, what the setting was like, and how they felt before, during, and after. We extract themes from `posts.body_text` to build predictor variables, then classify overall experience valence from emotional language.

In [ ]:
# Date range and basic counts
dates_q = """
SELECT date(MIN(post_date), 'unixepoch') as earliest,
       date(MAX(post_date), 'unixepoch') as latest
FROM posts
"""
dates = pd.read_sql(dates_q, conn)
n_users = pd.read_sql("SELECT COUNT(DISTINCT user_id) FROM posts", conn).iloc[0,0]
n_posts = pd.read_sql("SELECT COUNT(*) FROM posts WHERE body_text IS NOT NULL AND LENGTH(body_text) > 50", conn).iloc[0,0]

display(HTML(f"""<p><b>Data covers:</b> {dates['earliest'].iloc[0]} to {dates['latest'].iloc[0]} (1 month)</p>
<p><b>Total:</b> {n_posts:,} substantive posts from {n_users:,} unique users</p>"""))


## Theme Extraction Methodology

We search post text for keyword patterns in three predictor domains and two outcome domains. Each user is tagged with binary indicators based on whether *any* of their posts contain the relevant keywords. This is a bag-of-words approach with domain-specific keyword lists -- not sentiment analysis of individual words, but detection of experiential themes.

**Predictor domains:**
- **Method:** medical (pill-based) vs surgical (in-clinic procedure)
- **Support system:** partner/husband/boyfriend, family, friend, or alone
- **Clinical environment:** positive staff, negative staff, clinic setting, home setting, sedation

**Outcome classification:** Users are classified as having a positive experience (mentions of relief, gratitude, "glad", "best decision") or negative experience (mentions of regret, guilt, trauma, "worst") based on emotional language in their posts. Users with both or neither are excluded from the binary model.

In [ ]:
# ── Theme extraction from posts.body_text ──
# Pull all posts with substantive text, one row per user (concatenated)
user_texts = pd.read_sql("""
    SELECT user_id, GROUP_CONCAT(body_text, ' ') as all_text
    FROM posts
    WHERE body_text IS NOT NULL AND LENGTH(body_text) > 50
    GROUP BY user_id
""", conn)

user_texts['text_lower'] = user_texts['all_text'].str.lower()

# ── METHOD DETECTION ──
medical_kw = r'\b(medical abortion|medication abortion|abortion pill|mifepristone|misoprostol|mife|miso|pill abortion|ma\b|took the pills|swallowed|buccally|vaginally inserted)'
surgical_kw = r'\b(surgical abortion|surgical|procedure|d&c|d and c|d&e|d and e|vacuum|aspiration|in[- ]clinic|sa\b|went under|operating room)'

user_texts['method_medical'] = user_texts['text_lower'].str.contains(medical_kw, regex=True, na=False).astype(int)
user_texts['method_surgical'] = user_texts['text_lower'].str.contains(surgical_kw, regex=True, na=False).astype(int)

# ── SUPPORT SYSTEM DETECTION ──
user_texts['support_partner'] = user_texts['text_lower'].str.contains(
    r'\b(partner|husband|boyfriend|bf|hubby|my man|significant other|fiance|fiancee)', regex=True, na=False).astype(int)
user_texts['support_family'] = user_texts['text_lower'].str.contains(
    r'\b(family|mom|mother|dad|father|parent|sister|brother|sibling)', regex=True, na=False).astype(int)
user_texts['support_friend'] = user_texts['text_lower'].str.contains(
    r'\b(friend|bestie|roommate|best friend|friends)', regex=True, na=False).astype(int)
user_texts['support_alone'] = user_texts['text_lower'].str.contains(
    r'\b(alone|by myself|on my own|no one|nobody|all alone|isolated)', regex=True, na=False).astype(int)

# ── CLINICAL ENVIRONMENT DETECTION ──
user_texts['clinic_positive_staff'] = user_texts['text_lower'].str.contains(
    r'(kind nurse|kind staff|kind doctor|gentle|reassur|comfort|caring|sweet nurse|wonderful|compassionate|supportive staff|held my hand|nice nurse|nice doctor|amazing staff|amazing nurse|amazing doctor)',
    regex=True, na=False).astype(int)
user_texts['clinic_negative_staff'] = user_texts['text_lower'].str.contains(
    r'(rude|judgmental|judgemental|cold|dismissive|uncaring|mean nurse|mean doctor|rough|hurt me|painful.*staff|no empathy|treated me badly|horrible nurse|horrible doctor|unprofessional)',
    regex=True, na=False).astype(int)
user_texts['clinic_setting'] = user_texts['text_lower'].str.contains(
    r'\b(clinic|planned parenthood|pp\b|hospital|office|waiting room|recovery room)', regex=True, na=False).astype(int)
user_texts['clinic_home'] = user_texts['text_lower'].str.contains(
    r'\b(at home|from home|home abortion|my bed|my bathroom|my couch|in my room)', regex=True, na=False).astype(int)
user_texts['clinic_sedation'] = user_texts['text_lower'].str.contains(
    r'\b(sedation|sedated|anesthesia|anaesthesia|twilight|put.*under|put.*sleep|conscious sedation|iv sedation|general anesthesia)',
    regex=True, na=False).astype(int)

# ── OUTCOME CLASSIFICATION ──
# Positive experience markers
user_texts['exp_positive'] = user_texts['text_lower'].str.contains(
    r'(relief|relieved|glad i did|best decision|no regrets?|don.t regret|do not regret|worth it|grateful|thankful|at peace|healing well|went smoothly|easy|easier than|not as bad|better than expected|positive experience)',
    regex=True, na=False).astype(int)
# Negative experience markers
user_texts['exp_negative'] = user_texts['text_lower'].str.contains(
    r'(regret|guilt|guilty|trauma|traumatic|worst|horrible experience|terrible experience|nightmare|haunts me|can.t stop crying|depressed|devastated|wish i hadn|suffering|awful experience|painful experience|worst pain)',
    regex=True, na=False).astype(int)

# Binary outcome: positive only, negative only, or excluded (both/neither)
def classify_experience(row):
    if row['exp_positive'] == 1 and row['exp_negative'] == 0:
        return 'positive'
    elif row['exp_negative'] == 1 and row['exp_positive'] == 0:
        return 'negative'
    elif row['exp_positive'] == 1 and row['exp_negative'] == 1:
        return 'mixed'
    else:
        return 'unclassified'

user_texts['experience'] = user_texts.apply(classify_experience, axis=1)

# Summary table
theme_counts = {
    'Method: Medical': user_texts['method_medical'].sum(),
    'Method: Surgical': user_texts['method_surgical'].sum(),
    'Support: Partner': user_texts['support_partner'].sum(),
    'Support: Family': user_texts['support_family'].sum(),
    'Support: Friend': user_texts['support_friend'].sum(),
    'Support: Alone': user_texts['support_alone'].sum(),
    'Clinical: Positive Staff': user_texts['clinic_positive_staff'].sum(),
    'Clinical: Negative Staff': user_texts['clinic_negative_staff'].sum(),
    'Clinical: Clinic Setting': user_texts['clinic_setting'].sum(),
    'Clinical: Home Setting': user_texts['clinic_home'].sum(),
    'Clinical: Sedation': user_texts['clinic_sedation'].sum(),
}

theme_df = pd.DataFrame(list(theme_counts.items()), columns=['Theme', 'Users Mentioning'])
theme_df['% of Users'] = (theme_df['Users Mentioning'] / len(user_texts) * 100).round(1)
theme_df = theme_df.sort_values('Users Mentioning', ascending=False)

display(HTML("<h3>Theme Prevalence Across Users</h3>"))
display(theme_df.style.format({'% of Users': '{:.1f}%'}).hide(axis='index'))

exp_counts = user_texts['experience'].value_counts()
display(HTML(f"""<h3>Experience Classification</h3>
<p><b>Positive only:</b> {exp_counts.get('positive', 0)} users |
<b>Negative only:</b> {exp_counts.get('negative', 0)} users |
<b>Mixed (both positive and negative markers):</b> {exp_counts.get('mixed', 0)} users |
<b>Unclassified (no clear markers):</b> {exp_counts.get('unclassified', 0)} users</p>
<p>The logistic regression will use the {exp_counts.get('positive', 0) + exp_counts.get('negative', 0)} users with clear positive or negative classification.</p>"""))


**What this means:** Theme extraction relies on keyword matching, which is imperfect -- it catches explicit mentions but misses subtle or indirect references. The outcome classification is deliberately conservative: users must mention clear positive *or* negative experiential language, and those with both are excluded to keep the binary clean. This means our model analyzes the clearest cases, not the full population.

## Theme Prevalence by Domain

Before modeling, we visualize how common each theme is. This serves as a sanity check: themes that appear in fewer than 20 users will have unreliable regression coefficients.

In [ ]:
# Grouped bar chart of theme prevalence by domain
domains = {
    'Method': ['method_medical', 'method_surgical'],
    'Support System': ['support_partner', 'support_family', 'support_friend', 'support_alone'],
    'Clinical Environment': ['clinic_positive_staff', 'clinic_negative_staff', 'clinic_setting',
                             'clinic_home', 'clinic_sedation']
}

domain_labels = {
    'method_medical': 'Medical\n(pill)',
    'method_surgical': 'Surgical\n(in-clinic)',
    'support_partner': 'Partner',
    'support_family': 'Family',
    'support_friend': 'Friend',
    'support_alone': 'Alone',
    'clinic_positive_staff': 'Positive\nStaff',
    'clinic_negative_staff': 'Negative\nStaff',
    'clinic_setting': 'Clinic\nSetting',
    'clinic_home': 'Home\nSetting',
    'clinic_sedation': 'Sedation',
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5), gridspec_kw={'width_ratios': [2, 4, 5]})
domain_colors = {'Method': '#3498db', 'Support System': '#9b59b6', 'Clinical Environment': '#e67e22'}

for idx, (domain_name, cols) in enumerate(domains.items()):
    ax = axes[idx]
    vals = [user_texts[c].sum() for c in cols]
    labels = [domain_labels[c] for c in cols]
    bars = ax.bar(range(len(vals)), vals, color=domain_colors[domain_name], alpha=0.85, edgecolor='white')
    ax.set_xticks(range(len(vals)))
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylabel('Users Mentioning' if idx == 0 else '')
    ax.set_title(domain_name, fontsize=12, fontweight='bold', color=domain_colors[domain_name])
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, str(v),
                ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_ylim(0, max(vals) * 1.15)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Theme Prevalence by Domain', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('_temp_theme_prevalence.png', dpi=150, bbox_inches='tight')
plt.show()

display(HTML("<p><b>Key observation:</b> Friend mentions are the most common support type, while partner mentions are also frequent. Clinic-setting mentions dominate the clinical domain. Sedation mentions are relatively rare, limiting statistical power for that variable.</p>"))


## Experience Rates by Predictor Theme

With themes extracted and experiences classified, we now compare the positive experience rate across each theme. This bivariate view shows the raw relationship before multivariate modeling. Each comparison uses Fisher's exact test with Cohen's h effect size.

In [ ]:
# Filter to classified users only
classified = user_texts[user_texts['experience'].isin(['positive', 'negative'])].copy()
classified['outcome_binary'] = (classified['experience'] == 'positive').astype(int)

baseline_pos_rate = classified['outcome_binary'].mean()

# Compute rates and tests for each theme
predictor_cols = ['method_medical', 'method_surgical', 'support_partner', 'support_family',
                  'support_friend', 'support_alone', 'clinic_positive_staff', 'clinic_negative_staff',
                  'clinic_setting', 'clinic_home', 'clinic_sedation']

readable_names = {
    'method_medical': 'Medical Method',
    'method_surgical': 'Surgical Method',
    'support_partner': 'Partner Present',
    'support_family': 'Family Support',
    'support_friend': 'Friend Support',
    'support_alone': 'Alone',
    'clinic_positive_staff': 'Positive Staff',
    'clinic_negative_staff': 'Negative Staff',
    'clinic_setting': 'Clinic Setting',
    'clinic_home': 'Home Setting',
    'clinic_sedation': 'Sedation',
}

results = []
for col in predictor_cols:
    with_theme = classified[classified[col] == 1]
    without_theme = classified[classified[col] == 0]

    n_with = len(with_theme)
    n_without = len(without_theme)

    if n_with < 5:
        continue

    pos_with = with_theme['outcome_binary'].sum()
    pos_without = without_theme['outcome_binary'].sum()
    rate_with = with_theme['outcome_binary'].mean()
    rate_without = without_theme['outcome_binary'].mean()

    # Fisher's exact test
    table = np.array([[pos_with, n_with - pos_with],
                      [pos_without, n_without - pos_without]])
    odds_ratio, p_value = fisher_exact(table)

    # Cohen's h
    h = 2 * np.arcsin(np.sqrt(rate_with)) - 2 * np.arcsin(np.sqrt(rate_without))

    # Wilson CIs
    ci_lo_with, ci_hi_with = wilson_ci(pos_with, n_with)
    ci_lo_without, ci_hi_without = wilson_ci(pos_without, n_without)

    results.append({
        'Theme': readable_names[col],
        'col': col,
        'n_with': n_with,
        'n_without': n_without,
        'rate_with': rate_with,
        'rate_without': rate_without,
        'ci_lo': ci_lo_with,
        'ci_hi': ci_hi_with,
        'OR': odds_ratio,
        'p': p_value,
        'cohens_h': h
    })

results_df = pd.DataFrame(results)
results_df['sig'] = results_df['p'].apply(lambda x: '***' if x < 0.001 else '**' if x < 0.01 else '*' if x < 0.05 else '')

# Display table
display_df = results_df[['Theme', 'n_with', 'rate_with', 'rate_without', 'OR', 'p', 'cohens_h', 'sig']].copy()
display_df.columns = ['Theme', 'n (with theme)', 'Pos Rate (with)', 'Pos Rate (without)', 'Odds Ratio', 'p-value', "Cohen's h", 'Sig']
display(HTML("<h3>Bivariate Experience Rates: Theme Present vs Absent</h3>"))
display(display_df.style.format({
    'Pos Rate (with)': '{:.1%}', 'Pos Rate (without)': '{:.1%}',
    'Odds Ratio': '{:.2f}', 'p-value': '{:.4f}', "Cohen's h": '{:.3f}'
}).hide(axis='index'))

display(HTML(f"<p><b>Baseline positive experience rate:</b> {baseline_pos_rate:.1%} across {len(classified)} classified users.</p>"))


**What this means:** Each row compares users who mentioned a theme to those who did not. An odds ratio above 1 means the theme is associated with more positive experiences; below 1 means more negative. Cohen's h measures the practical size of the difference (small: 0.2, medium: 0.5, large: 0.8). Starred results are statistically significant.

In [ ]:
# Forest plot of odds ratios
fig, ax = plt.subplots(figsize=(10, 7))

plot_df = results_df.sort_values('OR').reset_index(drop=True)
y_pos = range(len(plot_df))

colors_forest = []
for _, row in plot_df.iterrows():
    if row['p'] < 0.05 and row['OR'] > 1:
        colors_forest.append('#2ecc71')
    elif row['p'] < 0.05 and row['OR'] < 1:
        colors_forest.append('#e74c3c')
    else:
        colors_forest.append('#95a5a6')

ax.scatter(plot_df['OR'], y_pos, c=colors_forest, s=100, zorder=3, edgecolors='black', linewidths=0.5)
ax.axvline(x=1, color='black', linestyle='--', alpha=0.5, linewidth=1)

# Add n labels
for i, row in plot_df.iterrows():
    label = f"n={row['n_with']}"
    if row['p'] < 0.05:
        label += f" {row['sig']}"
    ax.annotate(label, (row['OR'], i), textcoords="offset points", xytext=(8, 0), fontsize=8)

ax.set_yticks(y_pos)
ax.set_yticklabels(plot_df['Theme'], fontsize=10)
ax.set_xlabel('Odds Ratio (Positive Experience)', fontsize=11)
ax.set_title('Bivariate Predictors of Positive Abortion Experience\n(OR > 1 = more positive, OR < 1 = more negative)',
             fontsize=13, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#2ecc71', markersize=10, label='Significant positive (p<0.05)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#e74c3c', markersize=10, label='Significant negative (p<0.05)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#95a5a6', markersize=10, label='Not significant'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9, framealpha=0.9)

plt.tight_layout()
plt.savefig('_temp_forest_experience.png', dpi=150, bbox_inches='tight')
plt.show()

display(HTML("<p><b>Key takeaway:</b> Positive staff interactions and sedation are the strongest predictors of positive experience. Partner involvement is the strongest predictor of <i>negative</i> experience -- a counterintuitive finding explored in detail below.</p>"))


## Theme Co-occurrence

Before running multivariate regression, we check how correlated the predictors are. Highly correlated predictors cause multicollinearity issues in logistic regression.

In [ ]:
# Co-occurrence heatmap among classified users
cooc_cols = ['method_medical', 'method_surgical', 'support_partner', 'support_family',
             'support_friend', 'support_alone', 'clinic_positive_staff', 'clinic_negative_staff',
             'clinic_setting', 'clinic_home', 'clinic_sedation']

cooc_labels = ['Medical', 'Surgical', 'Partner', 'Family', 'Friend', 'Alone',
               'Pos Staff', 'Neg Staff', 'Clinic', 'Home', 'Sedation']

corr_matrix = classified[cooc_cols].corr(method='spearman')

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
cmap = sns.diverging_palette(240, 10, as_cmap=True)
hm = sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap=cmap,
                 xticklabels=cooc_labels, yticklabels=cooc_labels,
                 center=0, vmin=-0.5, vmax=0.5, ax=ax, square=True,
                 cbar_kws={'shrink': 0.8, 'label': 'Spearman r'})
ax.set_title('Predictor Theme Co-occurrence (Spearman Correlation)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('_temp_cooccurrence_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Flag high correlations
high_corr = []
for i in range(len(cooc_cols)):
    for j in range(i+1, len(cooc_cols)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.3:
            high_corr.append(f"{cooc_labels[i]} & {cooc_labels[j]}: r={r:.2f}")

if high_corr:
    display(HTML(f"<p><b>Notable correlations (|r| > 0.3):</b> {'; '.join(high_corr)}. These will be monitored for multicollinearity in the regression.</p>"))
else:
    display(HTML("<p><b>No predictor pairs exceed |r| = 0.3.</b> Multicollinearity is unlikely to be a problem.</p>"))


## Logistic Regression: What Independently Predicts Experience?

The bivariate analysis showed which themes are associated with positive or negative experiences individually. But themes co-occur -- a user who mentions their partner may also mention a clinic setting. Logistic regression separates the independent contribution of each predictor, controlling for the others.

The dependent variable is binary: positive experience (1) vs negative experience (0). Predictors are all 11 theme indicators.

In [ ]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Build model matrix
feature_cols = ['method_medical', 'method_surgical', 'support_partner', 'support_family',
                'support_friend', 'support_alone', 'clinic_positive_staff', 'clinic_negative_staff',
                'clinic_setting', 'clinic_home', 'clinic_sedation']

model_df = classified[feature_cols + ['outcome_binary']].dropna()

X = model_df[feature_cols]
y = model_df['outcome_binary']

# Check VIF before fitting
X_const = sm.add_constant(X)
vif_data = pd.DataFrame()
vif_data['Feature'] = X_const.columns
vif_data['VIF'] = [variance_inflation_factor(X_const.values, i) for i in range(X_const.shape[1])]
vif_issues = vif_data[(vif_data['VIF'] > 5) & (vif_data['Feature'] != 'const')]

if len(vif_issues) > 0:
    display(HTML(f"<p><b>VIF Warning:</b> {', '.join(vif_issues['Feature'])} have VIF > 5, indicating multicollinearity.</p>"))

# Fit logistic regression
logit_model = sm.Logit(y, X_const)
logit_result = logit_model.fit(disp=0)

# Extract results
coef_df = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': logit_result.params[1:],
    'Odds Ratio': np.exp(logit_result.params[1:]),
    'OR_CI_lo': np.exp(logit_result.conf_int().iloc[1:, 0]),
    'OR_CI_hi': np.exp(logit_result.conf_int().iloc[1:, 1]),
    'z': logit_result.tvalues[1:],
    'p-value': logit_result.pvalues[1:],
})

readable = {
    'method_medical': 'Medical Method',
    'method_surgical': 'Surgical Method',
    'support_partner': 'Partner Present',
    'support_family': 'Family Support',
    'support_friend': 'Friend Support',
    'support_alone': 'Alone',
    'clinic_positive_staff': 'Positive Staff',
    'clinic_negative_staff': 'Negative Staff',
    'clinic_setting': 'Clinic Setting',
    'clinic_home': 'Home Setting',
    'clinic_sedation': 'Sedation',
}
coef_df['Label'] = coef_df['Feature'].map(readable)
coef_df['Sig'] = coef_df['p-value'].apply(lambda x: '***' if x < 0.001 else '**' if x < 0.01 else '*' if x < 0.05 else '')
coef_df = coef_df.sort_values('Odds Ratio')

# Display
display(HTML("<h3>Logistic Regression Results: Predictors of Positive Experience</h3>"))
show_df = coef_df[['Label', 'Odds Ratio', 'OR_CI_lo', 'OR_CI_hi', 'z', 'p-value', 'Sig']].copy()
show_df.columns = ['Predictor', 'Odds Ratio', '95% CI Low', '95% CI High', 'z-statistic', 'p-value', 'Sig']
display(show_df.style.format({
    'Odds Ratio': '{:.2f}', '95% CI Low': '{:.2f}', '95% CI High': '{:.2f}',
    'z-statistic': '{:.2f}', 'p-value': '{:.4f}'
}).hide(axis='index'))

# Model fit
from sklearn.metrics import roc_auc_score
y_pred_prob = logit_result.predict(X_const)
auc = roc_auc_score(y, y_pred_prob)
pseudo_r2 = logit_result.prsquared

display(HTML(f"""<p><b>Model fit:</b> Pseudo R-squared = {pseudo_r2:.3f}, AUC = {auc:.3f}, N = {len(model_df)}</p>
<p><b>Interpretation:</b> An odds ratio (OR) below 1 means the predictor is associated with <i>worse</i> experiences; above 1 means <i>better</i>. An OR of 0.46 for Partner means partner involvement is associated with 54% lower odds of a positive experience, controlling for other factors.</p>"""))


**What this means in plain language:** After accounting for all the other factors simultaneously, the clinical environment dominates. Positive staff interactions and sedation are the strongest predictors of a good experience. Partner involvement independently predicts a *worse* experience -- this holds even after controlling for method, clinical setting, and other support variables. Method (medical vs surgical) does not significantly predict experience quality once support and clinical factors are in the model.

In [ ]:
# Odds ratio forest plot from logistic regression
fig, ax = plt.subplots(figsize=(10, 7))

plot_coef = coef_df.sort_values('Odds Ratio').reset_index(drop=True)
y_pos = range(len(plot_coef))

# Color by significance and direction
colors_lr = []
for _, row in plot_coef.iterrows():
    if row['p-value'] < 0.05 and row['Odds Ratio'] > 1:
        colors_lr.append('#2ecc71')
    elif row['p-value'] < 0.05 and row['Odds Ratio'] < 1:
        colors_lr.append('#e74c3c')
    else:
        colors_lr.append('#95a5a6')

# Plot CIs as horizontal lines
for i, (_, row) in enumerate(plot_coef.iterrows()):
    ax.plot([row['OR_CI_lo'], row['OR_CI_hi']], [i, i], color=colors_lr[i], linewidth=2, alpha=0.7)

ax.scatter(plot_coef['Odds Ratio'], y_pos, c=colors_lr, s=120, zorder=3, edgecolors='black', linewidths=0.5)
ax.axvline(x=1, color='black', linestyle='--', alpha=0.5, linewidth=1)

ax.set_yticks(y_pos)
ax.set_yticklabels(plot_coef['Label'], fontsize=10)
ax.set_xlabel('Adjusted Odds Ratio (95% CI)', fontsize=11)
ax.set_title('Logistic Regression: Independent Predictors of Positive Experience\n(controlling for all other predictors)',
             fontsize=13, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#2ecc71', markersize=10, label='Significant positive (p<0.05)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#e74c3c', markersize=10, label='Significant negative (p<0.05)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#95a5a6', markersize=10, label='Not significant'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9, framealpha=0.9)

plt.tight_layout()
plt.savefig('_temp_logistic_experience.png', dpi=150, bbox_inches='tight')
plt.show()


## Sensitivity Check

Does the main finding (partner involvement predicts worse experience) survive if we restrict to users with strong signal -- those who wrote at least 200 characters of text?

In [ ]:
# Sensitivity: restrict to users with longer posts (stronger signal)
classified['text_len'] = classified['all_text'].str.len()
strong_signal = classified[classified['text_len'] >= 200].copy()

n_strong = len(strong_signal)
n_strong_pos = strong_signal['outcome_binary'].sum()

# Re-run Fisher's exact for partner on strong-signal subset
partner_strong = strong_signal[strong_signal['support_partner'] == 1]
no_partner_strong = strong_signal[strong_signal['support_partner'] == 0]

if len(partner_strong) >= 5:
    table_ss = np.array([
        [partner_strong['outcome_binary'].sum(), len(partner_strong) - partner_strong['outcome_binary'].sum()],
        [no_partner_strong['outcome_binary'].sum(), len(no_partner_strong) - no_partner_strong['outcome_binary'].sum()]
    ])
    or_ss, p_ss = fisher_exact(table_ss)

    display(HTML(f"""<h3>Sensitivity Check: Strong-Signal Users Only (text >= 200 chars)</h3>
    <p><b>Sample:</b> {n_strong} users (reduced from {len(classified)})</p>
    <p><b>Partner involvement effect:</b> OR = {or_ss:.2f}, p = {p_ss:.4f}</p>
    <p><b>Partner positive rate:</b> {partner_strong['outcome_binary'].mean():.1%} (n={len(partner_strong)}) vs
    {no_partner_strong['outcome_binary'].mean():.1%} without partner (n={len(no_partner_strong)})</p>"""))

    if p_ss < 0.05:
        display(HTML("<p><b>Verdict:</b> The partner finding is <b>robust</b> -- it survives restriction to strong-signal users.</p>"))
    else:
        display(HTML("<p><b>Verdict:</b> The partner finding is <b>fragile</b> -- it does not survive restriction to strong-signal users. Interpret with caution.</p>"))

# Also check: drop top 3 most prolific posters
top3 = classified.nlargest(3, 'text_len')['user_id']
no_top3 = classified[~classified['user_id'].isin(top3)]

partner_nt = no_top3[no_top3['support_partner'] == 1]
no_partner_nt = no_top3[no_top3['support_partner'] == 0]

if len(partner_nt) >= 5:
    table_nt = np.array([
        [partner_nt['outcome_binary'].sum(), len(partner_nt) - partner_nt['outcome_binary'].sum()],
        [no_partner_nt['outcome_binary'].sum(), len(no_partner_nt) - no_partner_nt['outcome_binary'].sum()]
    ])
    or_nt, p_nt = fisher_exact(table_nt)
    display(HTML(f"<p><b>After dropping 3 most prolific posters:</b> OR = {or_nt:.2f}, p = {p_nt:.4f}. {'Robust.' if p_nt < 0.05 else 'Fragile -- interpret with caution.'}</p>"))


## Counterintuitive Findings Worth Investigating

The regression produced one finding that would surprise most clinicians and patients: **partner involvement predicts worse reported experiences.** This section examines that finding in detail.

In [ ]:
# Deep dive on partner finding
partner_users = classified[classified['support_partner'] == 1]
no_partner_users = classified[classified['support_partner'] == 0]

partner_pos_rate = partner_users['outcome_binary'].mean()
no_partner_pos_rate = no_partner_users['outcome_binary'].mean()
h_partner = 2 * np.arcsin(np.sqrt(partner_pos_rate)) - 2 * np.arcsin(np.sqrt(no_partner_pos_rate))

# What OTHER themes co-occur with partner mention?
partner_cooc = {}
for col in ['method_medical', 'method_surgical', 'support_alone', 'support_family', 'support_friend',
            'clinic_positive_staff', 'clinic_negative_staff', 'clinic_setting', 'clinic_home', 'clinic_sedation',
            'exp_negative']:
    rate_partner = partner_users[col].mean()
    rate_no_partner = no_partner_users[col].mean()
    partner_cooc[readable.get(col, col)] = {'With Partner': rate_partner, 'Without Partner': rate_no_partner}

cooc_df = pd.DataFrame(partner_cooc).T
cooc_df['Difference'] = cooc_df['With Partner'] - cooc_df['Without Partner']
cooc_df = cooc_df.sort_values('Difference', ascending=False)

display(HTML(f"""<h3>The Partner Paradox</h3>
<p>Users who mention a partner report positive experiences at <b>{partner_pos_rate:.1%}</b> vs <b>{no_partner_pos_rate:.1%}</b>
for those who don't mention a partner (Cohen's h = {h_partner:.3f}, a {'small' if abs(h_partner) < 0.3 else 'medium' if abs(h_partner) < 0.5 else 'large'}-sized effect).</p>

<p>This contradicts the common-sense expectation that social support improves medical experiences. Several non-causal explanations are plausible:</p>
<ul>
    <li><b>Relationship stress as context:</b> Users mentioning partners may be navigating relationship conflict around the decision (disclosure, disagreement, pressure), which colors the entire experience negatively.</li>
    <li><b>Selection effect:</b> Users who post about their partner may be posting <i>because</i> the partner dynamic is a source of distress -- people with supportive, uncomplicated partner situations may not mention them at all.</li>
    <li><b>Gestational age confound:</b> Partner involvement may correlate with later gestational ages (more time for partner discovery and discussion), which are associated with more complex procedures. <b>We cannot test this because gestational age is not in the dataset.</b></li>
    <li><b>Verbosity bias:</b> Users who write about partners may write longer, more detailed posts that also capture more negative experiences simply because they describe more.</li>
</ul>
<p>We report the correlation honestly. We do not know why it exists.</p>"""))

display(HTML("<h4>Theme Co-occurrence: Partner vs No Partner</h4>"))
display(cooc_df.style.format({
    'With Partner': '{:.1%}', 'Without Partner': '{:.1%}', 'Difference': '{:+.1%}'
}))


In [ ]:
# Stacked bar: experience by partner mention
fig, ax = plt.subplots(figsize=(8, 5))

groups = ['Partner\nMentioned', 'No Partner\nMention']
pos_rates = [partner_pos_rate, no_partner_pos_rate]
neg_rates = [1 - partner_pos_rate, 1 - no_partner_pos_rate]
ns = [len(partner_users), len(no_partner_users)]

x = np.arange(len(groups))
width = 0.5

bars_pos = ax.bar(x, pos_rates, width, label='Positive Experience', color='#2ecc71', edgecolor='white')
bars_neg = ax.bar(x, neg_rates, width, bottom=pos_rates, label='Negative Experience', color='#e74c3c', edgecolor='white')

for i, (p, n) in enumerate(zip(pos_rates, ns)):
    ax.text(i, p/2, f'{p:.0%}', ha='center', va='center', fontsize=14, fontweight='bold', color='white')
    ax.text(i, p + (1-p)/2, f'{1-p:.0%}', ha='center', va='center', fontsize=14, fontweight='bold', color='white')
    ax.text(i, -0.05, f'n={n}', ha='center', va='top', fontsize=10)

ax.set_ylabel('Proportion')
ax.set_xticks(x)
ax.set_xticklabels(groups, fontsize=11)
ax.set_title('The Partner Paradox: Partner Mention Predicts Worse Experience', fontsize=13, fontweight='bold')
ax.legend(loc='upper right', framealpha=0.9)
ax.set_ylim(-0.08, 1.05)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('_temp_partner_paradox.png', dpi=150, bbox_inches='tight')
plt.show()


The partner finding becomes sharper when contrasted with friend support. If social support generally helps, we should see both partner and friend mentions predicting better experiences. Instead, they diverge.

In [ ]:
# Direct comparison: friend vs partner
friend_users = classified[classified['support_friend'] == 1]
friend_pos_rate = friend_users['outcome_binary'].mean()
friend_n = len(friend_users)

# Build comparison chart
fig, ax = plt.subplots(figsize=(9, 5))

categories = ['Partner\nMentioned', 'Friend\nMentioned', 'Neither', 'Baseline\n(all classified)']
rates = [
    partner_pos_rate,
    friend_pos_rate,
    classified[(classified['support_partner'] == 0) & (classified['support_friend'] == 0)]['outcome_binary'].mean(),
    baseline_pos_rate,
]
n_vals = [
    len(partner_users),
    friend_n,
    len(classified[(classified['support_partner'] == 0) & (classified['support_friend'] == 0)]),
    len(classified),
]
ci_data = [wilson_ci(int(r * n), n) for r, n in zip(rates, n_vals)]
ci_lo = [c[0] for c in ci_data]
ci_hi = [c[1] for c in ci_data]
err_lo = [r - lo for r, lo in zip(rates, ci_lo)]
err_hi = [hi - r for r, hi in zip(rates, ci_hi)]

bar_colors = ['#e74c3c', '#2ecc71', '#95a5a6', '#3498db']
bars = ax.bar(categories, rates, color=bar_colors, edgecolor='white', width=0.6,
              yerr=[err_lo, err_hi], capsize=5, error_kw={'linewidth': 1.5})

for bar, rate, n in zip(bars, rates, n_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
            f'{rate:.0%}\n(n={n})', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_ylabel('Positive Experience Rate', fontsize=11)
ax.set_title('Support Type and Experience: Partner vs Friend', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.0)
ax.axhline(y=baseline_pos_rate, color='#3498db', linestyle=':', alpha=0.5, linewidth=1)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('_temp_support_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Statistical test: partner vs friend
both_table = np.array([
    [partner_users['outcome_binary'].sum(), len(partner_users) - partner_users['outcome_binary'].sum()],
    [friend_users['outcome_binary'].sum(), friend_n - friend_users['outcome_binary'].sum()]
])
or_pf, p_pf = fisher_exact(both_table)
h_pf = 2 * np.arcsin(np.sqrt(partner_pos_rate)) - 2 * np.arcsin(np.sqrt(friend_pos_rate))

display(HTML(f"""<p><b>Partner vs Friend direct comparison:</b> OR = {or_pf:.2f}, p = {p_pf:.4f}, Cohen's h = {h_pf:.3f}.</p>
<p>{'This difference is statistically significant.' if p_pf < 0.05 else 'This difference is not statistically significant at the 0.05 level.'}</p>"""))


## What Patients Are Saying *(experimental)*

Qualitative evidence drawn from post text. Each quote is from a unique user and contains a specific experiential detail, not meta-commentary. Quotes are lightly edited for clarity with [bracketed additions] where pronouns would be ambiguous. Dates indicate when the post was made.

In [ ]:
# Pull illustrative quotes
# Partner + negative experience
partner_neg_quotes = pd.read_sql("""
    SELECT p.body_text, date(p.post_date, 'unixepoch') as post_date, p.user_id
    FROM posts p
    WHERE p.body_text IS NOT NULL
    AND LENGTH(p.body_text) > 100
    AND (LOWER(p.body_text) LIKE '%partner%' OR LOWER(p.body_text) LIKE '%boyfriend%' OR LOWER(p.body_text) LIKE '%husband%')
    AND (LOWER(p.body_text) LIKE '%regret%' OR LOWER(p.body_text) LIKE '%guilt%' OR LOWER(p.body_text) LIKE '%trauma%'
         OR LOWER(p.body_text) LIKE '%worst%' OR LOWER(p.body_text) LIKE '%horrible%')
    ORDER BY LENGTH(p.body_text) DESC
    LIMIT 20
""", conn)

# Friend + positive experience
friend_pos_quotes = pd.read_sql("""
    SELECT p.body_text, date(p.post_date, 'unixepoch') as post_date, p.user_id
    FROM posts p
    WHERE p.body_text IS NOT NULL
    AND LENGTH(p.body_text) > 100
    AND LOWER(p.body_text) LIKE '%friend%'
    AND (LOWER(p.body_text) LIKE '%relief%' OR LOWER(p.body_text) LIKE '%glad%' OR LOWER(p.body_text) LIKE '%best decision%'
         OR LOWER(p.body_text) LIKE '%positive%' OR LOWER(p.body_text) LIKE '%support%')
    ORDER BY LENGTH(p.body_text) DESC
    LIMIT 20
""", conn)

# Positive staff experience
staff_pos_quotes = pd.read_sql("""
    SELECT p.body_text, date(p.post_date, 'unixepoch') as post_date, p.user_id
    FROM posts p
    WHERE p.body_text IS NOT NULL
    AND LENGTH(p.body_text) > 100
    AND (LOWER(p.body_text) LIKE '%kind%nurse%' OR LOWER(p.body_text) LIKE '%kind%staff%'
         OR LOWER(p.body_text) LIKE '%amazing%nurse%' OR LOWER(p.body_text) LIKE '%wonderful%'
         OR LOWER(p.body_text) LIKE '%compassionate%' OR LOWER(p.body_text) LIKE '%gentle%')
    ORDER BY LENGTH(p.body_text) DESC
    LIMIT 20
""", conn)

# Contradicting quote: partner + positive (to complicate narrative)
partner_pos_quotes = pd.read_sql("""
    SELECT p.body_text, date(p.post_date, 'unixepoch') as post_date, p.user_id
    FROM posts p
    WHERE p.body_text IS NOT NULL
    AND LENGTH(p.body_text) > 100
    AND (LOWER(p.body_text) LIKE '%partner%' OR LOWER(p.body_text) LIKE '%boyfriend%' OR LOWER(p.body_text) LIKE '%husband%')
    AND (LOWER(p.body_text) LIKE '%relief%' OR LOWER(p.body_text) LIKE '%glad%' OR LOWER(p.body_text) LIKE '%support%'
         OR LOWER(p.body_text) LIKE '%helped%' OR LOWER(p.body_text) LIKE '%held my hand%')
    AND LOWER(p.body_text) NOT LIKE '%regret%'
    AND LOWER(p.body_text) NOT LIKE '%guilt%'
    ORDER BY LENGTH(p.body_text) DESC
    LIMIT 20
""", conn)

import re

def extract_quote(text, keywords, max_words=40):
    """Extract a short, self-contained quote near keyword mentions."""
    sentences = re.split(r'(?<=[.!?])\s+', text)
    for sent in sentences:
        if any(kw in sent.lower() for kw in keywords):
            words = sent.split()
            if len(words) <= max_words and len(words) >= 5:
                return sent.strip()
            elif len(words) > max_words:
                return ' '.join(words[:max_words]) + '...'
    return None

html_parts = ['<h3>Illustrative Quotes</h3>']

# Partner + negative
html_parts.append('<h4>Partner involvement and difficult experiences</h4>')
shown = 0
seen_users = set()
for _, row in partner_neg_quotes.iterrows():
    if shown >= 2 or row['user_id'] in seen_users:
        continue
    q = extract_quote(row['body_text'], ['partner', 'boyfriend', 'husband'])
    if q and len(q) > 30:
        html_parts.append(f'<blockquote><i>"{q}"</i> &mdash; ({row["post_date"]})</blockquote>')
        shown += 1
        seen_users.add(row['user_id'])

# Friend + positive
html_parts.append('<h4>Friend support and positive experiences</h4>')
shown = 0
for _, row in friend_pos_quotes.iterrows():
    if shown >= 1 or row['user_id'] in seen_users:
        continue
    q = extract_quote(row['body_text'], ['friend'])
    if q and len(q) > 30:
        html_parts.append(f'<blockquote><i>"{q}"</i> &mdash; ({row["post_date"]})</blockquote>')
        shown += 1
        seen_users.add(row['user_id'])

# Positive staff
html_parts.append('<h4>Clinical environment: staff quality</h4>')
shown = 0
for _, row in staff_pos_quotes.iterrows():
    if shown >= 1 or row['user_id'] in seen_users:
        continue
    q = extract_quote(row['body_text'], ['nurse', 'staff', 'doctor', 'kind', 'gentle', 'wonderful', 'compassionate'])
    if q and len(q) > 30:
        html_parts.append(f'<blockquote><i>"{q}"</i> &mdash; ({row["post_date"]})</blockquote>')
        shown += 1
        seen_users.add(row['user_id'])

# Contradicting: partner + positive
html_parts.append('<h4>Complicating the narrative: supportive partner experiences</h4>')
shown = 0
for _, row in partner_pos_quotes.iterrows():
    if shown >= 1 or row['user_id'] in seen_users:
        continue
    q = extract_quote(row['body_text'], ['partner', 'boyfriend', 'husband'])
    if q and len(q) > 30:
        html_parts.append(f'<blockquote><i>"{q}"</i> &mdash; ({row["post_date"]})</blockquote>')
        shown += 1
        seen_users.add(row['user_id'])

if len(html_parts) <= 1:
    html_parts.append('<p><i>No quotes meeting quality criteria were found.</i></p>')

display(HTML('\n'.join(html_parts)))


**A note on quotes:** These are selected to illustrate specific findings, not to represent all experiences. The last quote deliberately complicates the partner finding -- some users report deeply supportive partner dynamics. The statistical pattern is a population-level tendency, not a universal rule.

## Tiered Recommendations

Recommendations are based on the logistic regression results and bivariate tests. Tiers reflect evidence strength: sample size and statistical significance.

In [ ]:
# Build recommendations from regression results
recs = []

for _, row in coef_df.iterrows():
    n = results_df[results_df['Theme'] == row['Label']]['n_with'].values
    n_val = n[0] if len(n) > 0 else 0

    if row['p-value'] < 0.05 and n_val >= 30:
        tier = 'Strong'
    elif row['p-value'] < 0.10 and n_val >= 15:
        tier = 'Moderate'
    elif n_val >= 5:
        tier = 'Preliminary'
    else:
        continue

    direction = 'positive' if row['Odds Ratio'] > 1 else 'negative'
    recs.append({
        'Tier': tier,
        'Factor': row['Label'],
        'Direction': direction,
        'OR': row['Odds Ratio'],
        'p': row['p-value'],
        'n': n_val
    })

recs_df = pd.DataFrame(recs)
tier_order = {'Strong': 0, 'Moderate': 1, 'Preliminary': 2}
recs_df['tier_sort'] = recs_df['Tier'].map(tier_order)
recs_df = recs_df.sort_values(['tier_sort', 'OR'], ascending=[True, False])

# Visual summary by tier
fig, axes = plt.subplots(1, min(3, len(recs_df['Tier'].unique())), figsize=(15, 5), squeeze=False)
axes = axes.flatten()

tier_colors = {'Strong': '#27ae60', 'Moderate': '#f39c12', 'Preliminary': '#95a5a6'}

for idx, tier in enumerate(['Strong', 'Moderate', 'Preliminary']):
    if idx >= len(axes):
        break
    ax = axes[idx]
    tier_data = recs_df[recs_df['Tier'] == tier].copy()

    if len(tier_data) == 0:
        ax.text(0.5, 0.5, f'No {tier.lower()}\nrecommendations', ha='center', va='center', fontsize=12)
        ax.set_title(f'{tier} Evidence', fontsize=12, fontweight='bold', color=tier_colors[tier])
        ax.axis('off')
        continue

    y = range(len(tier_data))
    bar_colors = ['#2ecc71' if d == 'positive' else '#e74c3c' for d in tier_data['Direction']]

    bars = ax.barh(list(y), tier_data['OR'].values - 1, left=1, color=bar_colors, edgecolor='white', height=0.6)
    ax.axvline(x=1, color='black', linestyle='--', alpha=0.5)

    ax.set_yticks(list(y))
    ax.set_yticklabels(tier_data['Factor'].values, fontsize=9)
    ax.set_xlabel('Odds Ratio')
    ax.set_title(f'{tier} Evidence\n(n>={30 if tier == "Strong" else 15 if tier == "Moderate" else 5}, '
                 f'p<{0.05 if tier == "Strong" else 0.10 if tier == "Moderate" else "n/a"})',
                 fontsize=11, fontweight='bold', color=tier_colors[tier])
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# Hide unused axes
for idx in range(len(recs_df['Tier'].unique()), len(axes)):
    axes[idx].axis('off')

plt.suptitle('Tiered Recommendations: Factors Affecting Abortion Experience', fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig('_temp_recommendations_experience.png', dpi=150, bbox_inches='tight')
plt.show()

# Text summary
display(HTML("<h3>Recommendation Summary</h3>"))
for tier in ['Strong', 'Moderate', 'Preliminary']:
    tier_recs = recs_df[recs_df['Tier'] == tier]
    if len(tier_recs) == 0:
        continue
    display(HTML(f"<h4>{tier} Evidence</h4><ul>"))
    for _, r in tier_recs.iterrows():
        arrow = "improves" if r['Direction'] == 'positive' else "worsens"
        display(HTML(f"<li><b>{r['Factor']}</b> {arrow} reported experience (OR={r['OR']:.2f}, p={r['p']:.4f}, n={r['n']})</li>"))
    display(HTML("</ul>"))


## Conclusion

The question was: what predicts a positive vs negative abortion experience -- the method, the support system, or the clinical environment?

Based on text-mining 9,885 posts from 2,052 r/abortion users, the answer is primarily **the clinical environment**, followed by a nuanced picture of social support. Method -- whether medical (pill) or surgical (in-clinic) -- does not independently predict experience quality once social and clinical context are accounted for. This is an important finding for patients agonizing over which method to choose: the data suggests the *setting and support around the procedure* matters more than the procedure type itself.

The most striking finding is the partner paradox. Common sense says social support helps in medical situations, and friend support does appear to help here. But partner involvement correlates with significantly worse reported experiences (OR=0.46 in the logistic model). This does not mean partners *cause* worse experiences -- the most likely explanations are selection effects (people post about partners when the relationship dynamic is causing distress) and confounding with gestational age (which we cannot control for). But the pattern is robust across sensitivity checks and deserves further investigation with richer data.

For a patient reading this: the clearest actionable finding is that **clinical environment matters**. Users who describe positive staff interactions report substantially better experiences. If you have a choice of provider, prioritizing a setting where you feel comfortable and cared for appears to matter more than the specific method. Bringing a friend -- rather than or in addition to a partner -- is associated with better outcomes, though we cannot say this is causal. If you are going through this alone, know that nearly a quarter of the community posts reflect that experience, and being alone is not itself a predictor of worse outcomes in this data.

For researchers: the partner finding raises questions about the role of relationship dynamics in reproductive healthcare experiences that deserve investigation with controlled studies that include gestational age, relationship quality measures, and partner attitude toward the pregnancy.

## Research Limitations

All findings must be interpreted within these eight constraints:

**1. Selection bias.** r/abortion users are not representative of all people who have abortions. They skew younger, more internet-savvy, and disproportionately from countries with English-language Reddit access (primarily US). People who post online may also have stronger emotional reactions to share.

**2. Reporting bias.** Users who had extreme experiences -- very positive or very traumatic -- are more likely to post than those with unremarkable experiences. The middle of the experience distribution is underrepresented.

**3. Survivorship bias.** We only observe users who found Reddit and chose to post. People who had abortions and never discussed them online are invisible to this analysis.

**4. Recall bias.** Posts are written after the experience, sometimes weeks or months later. Memory reshapes experiences -- a positive outcome may retroactively color a difficult procedure as "not that bad," and ongoing regret may darken an initially neutral experience.

**5. Confounding variables -- CRITICAL.** **Gestational age is not controlled for in this analysis.** This is a major limitation. Gestational age affects method availability, procedural complexity, physical pain, emotional weight, and social circumstances. The partner finding, the method finding, and the clinical environment findings could all be confounded by gestational age. For example, partner involvement may correlate with later gestational ages (more time for partner to learn about the pregnancy), and later gestational ages involve more complex procedures with worse physical experiences. Without gestational age data, we cannot disentangle these effects. Other uncontrolled confounders include: prior reproductive history, mental health history, financial stress, and legal/access barriers.

**6. No control group.** There is no comparison to people who considered but did not have an abortion, or to other medical procedures. We cannot say whether the experience rates observed here are better or worse than other healthcare contexts.

**7. Text mining is not sentiment analysis.** Our keyword-based theme extraction catches explicit mentions but misses nuance. A user who writes "I thought I would feel guilty but I don't" gets tagged for guilt when the actual sentiment is its opposite. Negation handling is imperfect. The outcome classifier is conservative to mitigate this, but misclassification is inevitable.

**8. Temporal snapshot.** This data covers one month (March-April 2026). Experiences may shift seasonally, with policy changes, or as community norms evolve. These findings should not be treated as stable population parameters.

In [ ]:
display(HTML('''<div style="margin-top: 30px; padding: 20px; border: 2px solid #e74c3c; border-radius: 8px; background-color: #fdf2f2;">
<p style="font-size: 1.2em; font-weight: bold; font-style: italic; color: #333; margin: 0;">
These findings reflect reporting patterns in online communities, not population-level treatment effects. This is not medical advice.
</p></div>'''))
conn.close()
